In [ ]:
# ============================================================
# Customer RFM Segmentation & Retention Dashboard
# Single-cell Google Colab Script (100% FIXED)
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# STEP 1: GENERATE REALISTIC E-COMMERCE DATA
# ─────────────────────────────────────────────
np.random.seed(42)

N_CUSTOMERS = 3000
N_TRANSACTIONS = 8000   # more transactions than customers → multiple orders

end_date   = pd.Timestamp('today').normalize()
start_date = end_date - pd.Timedelta(days=364)

# Draw random customer IDs
customer_ids = np.random.choice(
    [f"CUST_{i:04d}" for i in range(1, N_CUSTOMERS + 1)],
    size=N_TRANSACTIONS,
    replace=True
)

# Random order dates spread over last 365 days
random_days  = np.random.randint(0, 365, size=N_TRANSACTIONS)
order_dates  = [start_date + pd.Timedelta(days=int(d)) for d in random_days]

# Order values between 300 and 5000 EGP
order_values = np.round(np.random.uniform(300, 5000, size=N_TRANSACTIONS), 2)

df = pd.DataFrame({
    "Customer_ID"    : customer_ids,
    "Order_Date"     : order_dates,
    "Order_Value_EGP": order_values
})

print(f"✅ Generated {len(df):,} transactions for {df['Customer_ID'].nunique():,} unique customers.")

# ─────────────────────────────────────────────
# STEP 2: CALCULATE RFM METRICS & SEGMENTS
# ─────────────────────────────────────────────
analysis_date = df["Order_Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer_ID").agg(
    Last_Order_Date = ("Order_Date",     "max"),
    Frequency       = ("Order_Date",     "count"),
    Monetary        = ("Order_Value_EGP","sum")
).reset_index()

rfm["Recency"] = (analysis_date - rfm["Last_Order_Date"]).dt.days

# Score R, F, M (1–4; 4 = best)
rfm["R_Score"] = pd.qcut(rfm["Recency"],  q=4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], q=4, labels=[1, 2, 3, 4]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

def assign_segment(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]

    if r == 4 and f == 4 and m == 4: return "Champions"
    elif r >= 3 and f >= 3 and m >= 3: return "Champions"
    elif f >= 3 and m >= 3 and r >= 2: return "Loyal Customers"
    elif r <= 2 and f >= 3 and m >= 3: return "At Risk"
    elif r == 4 and f <= 2: return "New Customers"
    elif r == 3 and f <= 2: return "New Customers"
    elif r <= 2 and f <= 2 and m >= 3: return "Potential Churners"
    elif r >= 3 and f >= 2 and m <= 2: return "Promising"
    elif r == 2 and f == 2 and m == 2: return "Need Attention"
    elif r <= 2 and f <= 2 and m <= 2: return "Lost"
    else: return "Need Attention"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

print("\n📊 Segment Distribution:")
print(rfm["Segment"].value_counts().to_string())

# ─────────────────────────────────────────────
# STEP 3: CREATE 3 INDEPENDENT PLOTLY FIGURES
# ─────────────────────────────────────────────
PALETTE = {
    "Champions"         : "#BF5AF2",
    "Loyal Customers"   : "#FF375F",
    "At Risk"           : "#FF9F0A",
    "New Customers"     : "#30D158",
    "Potential Churners": "#64D2FF",
    "Promising"         : "#FFD60A",
    "Need Attention"    : "#FF6B35",
    "Lost"              : "#8E8E93",
}

LAYOUT_BASE = dict(
    paper_bgcolor="#0F0F1A",
    plot_bgcolor ="#0F0F1A",
    font         =dict(family="Inter, sans-serif", color="#E8E8F0", size=13),
)

seg_summary = rfm.groupby("Segment").agg(
    Count    = ("Customer_ID", "count"),
    Avg_Mon  = ("Monetary",    "mean"),
    Total_Mon= ("Monetary",    "sum"),
).reset_index()

# ── Figure 1: Treemap
fig_treemap = go.Figure(go.Treemap(
    labels        = seg_summary["Segment"],
    parents       = [""] * len(seg_summary),
    values        = seg_summary["Count"],
    customdata    = seg_summary["Avg_Mon"].round(0),
    marker_colors = [PALETTE.get(s, "#888") for s in seg_summary["Segment"]],
    texttemplate  = "<b>%{label}</b><br>%{value} customers<br>Avg: EGP %{customdata:,}",
    hovertemplate = "<b>%{label}</b><br>Customers: %{value}<br>Avg Monetary: EGP %{customdata:,}<extra></extra>",
))
fig_treemap.update_layout(
    **LAYOUT_BASE,
    title=dict(text="🗂 Customer Segments Overview", font=dict(size=20, color="#BF5AF2"), x=0.01),
    margin=dict(t=60, l=10, r=10, b=10),
)

# ── Figure 2: Scatter (BUG FIXED)
fig_scatter = go.Figure()
for seg, colour in PALETTE.items():
    subset = rfm[rfm["Segment"] == seg]
    if subset.empty:
        continue
    fig_scatter.add_trace(go.Scatter(
        x          = subset["Recency"],
        y          = subset["Frequency"],
        mode       = "markers",
        name       = seg,
        marker     = dict(
            color  = colour,
            size   = np.clip(subset["Monetary"] / 600, 6, 28),
            opacity= 0.75,
            line   = dict(width=0.5, color="rgba(255, 255, 255, 0.2)"), # <--- Fixed rgba color
        ),
        customdata = subset[["Monetary", "RFM_Score"]],
        hovertemplate=(
            f"<b>{seg}</b><br>"
            "Recency: %{x} days<br>"
            "Frequency: %{y} orders<br>"
            "Monetary: EGP %{customdata[0]:,.0f}<br>"
            "RFM Score: %{customdata[1]}<extra></extra>"
        ),
    ))
fig_scatter.update_layout(
    **LAYOUT_BASE,
    title =dict(text="📍 RFM Distribution: Recency vs Frequency", font=dict(size=18, color="#BF5AF2"), x=0.01),
    xaxis =dict(title="Recency (Days Since Last Order)", gridcolor="#1E1E3A", zeroline=False),
    yaxis =dict(title="Frequency (Number of Orders)",   gridcolor="#1E1E3A", zeroline=False),
    legend=dict(bgcolor="#1A1A2E", bordercolor="#3A3A5C", borderwidth=1, font=dict(size=11)),
    margin=dict(t=60, l=60, r=20, b=60),
)

# ── Figure 3: Bar
seg_bar = seg_summary.sort_values("Avg_Mon", ascending=False)
fig_bar = go.Figure(go.Bar(
    x             = seg_bar["Segment"],
    y             = seg_bar["Avg_Mon"].round(0),
    marker_color  = [PALETTE.get(s, "#888") for s in seg_bar["Segment"]],
    marker_line   = dict(width=0),
    text          = ["EGP {:,.0f}".format(v) for v in seg_bar["Avg_Mon"]],
    textposition  = "outside",
    textfont      = dict(color="#E8E8F0", size=11),
    hovertemplate = "<b>%{x}</b><br>Avg CLV: EGP %{y:,.0f}<extra></extra>",
))
fig_bar.update_layout(
    **LAYOUT_BASE,
    title =dict(text="💰 Avg Customer Lifetime Value by Segment", font=dict(size=18, color="#BF5AF2"), x=0.01),
    xaxis =dict(tickangle=-30, gridcolor="#1E1E3A"),
    yaxis =dict(title="Average Monetary (EGP)", gridcolor="#1E1E3A", zeroline=False),
    margin=dict(t=60, l=70, r=20, b=100),
    showlegend=False,
)

print("✅ All 3 Plotly figures created.")

# ─────────────────────────────────────────────
# STEP 4: BUILD HTML DASHBOARD
# ─────────────────────────────────────────────

# ── KPI Values
total_customers = rfm["Customer_ID"].nunique()
total_revenue   = rfm["Monetary"].sum()
pct_champions   = (rfm["Segment"] == "Champions").mean() * 100
pct_at_risk     = (rfm["Segment"] == "At Risk").mean()   * 100

treemap_html = fig_treemap.to_html(full_html=False, include_plotlyjs=False, config={"responsive": True})
scatter_html = fig_scatter.to_html(full_html=False, include_plotlyjs=False, config={"responsive": True})
bar_html     = fig_bar.to_html(    full_html=False, include_plotlyjs=False, config={"responsive": True})

html_template = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>E-Commerce RFM Dashboard</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <link rel="preconnect" href="https://fonts.googleapis.com"/>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap" rel="stylesheet"/>
  <style>
    *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ background: #0F0F1A; color: #E8E8F0; font-family: 'Inter', sans-serif; min-height: 100vh; padding: 24px; }}
    header {{ display: flex; align-items: center; gap: 16px; margin-bottom: 28px; }}
    .header-icon {{ font-size: 2.4rem; line-height: 1; }}
    .header-text h1 {{ font-size: 1.7rem; font-weight: 700; background: linear-gradient(90deg, #BF5AF2, #FF375F); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; }}
    .header-text p {{ font-size: 0.85rem; color: #8E8EA0; margin-top: 2px; }}
    .glass {{ background: rgba(255,255,255,0.04); border: 1px solid rgba(255,255,255,0.09); border-radius: 16px; backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px); }}
    .kpi-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin-bottom: 20px; }}
    @media (max-width: 900px) {{ .kpi-grid {{ grid-template-columns: repeat(2, 1fr); }} }}
    .kpi-card {{ padding: 22px 20px; display: flex; flex-direction: column; gap: 6px; transition: transform .2s, box-shadow .2s; }}
    .kpi-card:hover {{ transform: translateY(-3px); box-shadow: 0 8px 32px rgba(191,90,242,.18); }}
    .kpi-label {{ font-size: 0.75rem; font-weight: 600; letter-spacing: .08em; text-transform: uppercase; color: #8E8EA0; }}
    .kpi-value {{ font-size: 2rem; font-weight: 700; line-height: 1.1; }}
    .kpi-sub {{ font-size: 0.78rem; color: #6E6E80; }}
    .kpi-1 .kpi-value {{ color: #BF5AF2; }}
    .kpi-2 .kpi-value {{ color: #FF9F0A; }}
    .kpi-3 .kpi-value {{ color: #30D158; }}
    .kpi-4 .kpi-value {{ color: #FF375F; }}
    .chart-full {{ margin-bottom: 20px; padding: 10px; }}
    .chart-row {{ display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }}
    @media (max-width: 800px) {{ .chart-row {{ grid-template-columns: 1fr; }} }}
    .chart-card {{ padding: 10px; }}
    footer {{ text-align: center; color: #4A4A5A; font-size: 0.75rem; margin-top: 28px; padding-top: 16px; border-top: 1px solid rgba(255,255,255,0.06); }}
  </style>
</head>
<body>
  <header>
    <div class="header-icon">📊</div>
    <div class="header-text">
      <h1>E-Commerce RFM Segmentation Dashboard</h1>
      <p>Customer Retention Intelligence · Analysis Date: {analysis_date.strftime('%d %B %Y')}</p>
    </div>
  </header>
  <div class="kpi-grid">
    <div class="kpi-card glass kpi-1">
      <div class="kpi-label">Total Customers</div>
      <div class="kpi-value">{total_customers:,}</div>
      <div class="kpi-sub">Unique buyers in last 365 days</div>
    </div>
    <div class="kpi-card glass kpi-2">
      <div class="kpi-label">Total Revenue</div>
      <div class="kpi-value">EGP {total_revenue/1_000_000:.2f}M</div>
      <div class="kpi-sub">{total_revenue:,.0f} EGP gross</div>
    </div>
    <div class="kpi-card glass kpi-3">
      <div class="kpi-label">Champions</div>
      <div class="kpi-value">{pct_champions:.1f}%</div>
      <div class="kpi-sub">High-value loyal customers</div>
    </div>
    <div class="kpi-card glass kpi-4">
      <div class="kpi-label">At Risk</div>
      <div class="kpi-value">{pct_at_risk:.1f}%</div>
      <div class="kpi-sub">Need immediate re-engagement</div>
    </div>
  </div>
  <div class="chart-full glass">
    {treemap_html}
  </div>
  <div class="chart-row">
    <div class="chart-card glass">
      {scatter_html}
    </div>
    <div class="chart-card glass">
      {bar_html}
    </div>
  </div>
  <footer>
    Built with Python · Plotly · Pure HTML/CSS &nbsp;|&nbsp; © 2024 E-Commerce Analytics
  </footer>
</body>
</html>"""

# ─────────────────────────────────────────────
# STEP 5: EXPORT & DOWNLOAD
# ─────────────────────────────────────────────
output_path = "Ecommerce_RFM_Dashboard.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_template)

print(f"\n🎉 Dashboard saved → {output_path}")
print(f"   File size: {len(html_template)/1024:.1f} KB")

# Auto-download the file in Colab
try:
    from google.colab import files
    files.download(output_path)
    print("⬇️ Downloading file automatically...")
except Exception as e:
    print("Download handled locally or manually needed.")